The first class Bandit generates rewards produced by arms. 
In this class $T$ is the number of times the arms will be pulled. $N$ is the number of arms. 

When class initialised for the first time it generates randomly the means of arms. The means are stored in the array means[]. The rewards for  arm $i$  are generated from the Normal distribution N(mean[i],1) in generate\_arms. 
next(i):  returns the next reward of arm $i$. 
curr_arm(i):  returns how many times arm $i$ has been pullled 
proportions: returns array with proporions of times arms have been pulled. 
An instance of this class stores generated rewards. 
Reset: clears all generated rewards. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt,log

class Bandit:

    # T - number of time slots 
    # N - number of arms 


    def __init__(self, T, N): 
        self.T = T 
        self.N = N 
        self.curr =0
        # generate means of normal random variables 
        self.means = self.generate_means(N)
        # generate T normal random variable with the above means and unit variance for each arm 
        self.arms = self.generate_arms(N,T)
        # number of rewards of arm i that have been returned so far 
        self.curr_position = np.zeros(N, dtype=int)
        self.rewards = np.zeros(self.T+2)
        # average - array of averages. 
        self.averages = np.zeros(self.N)    

    def generate_means(self,N):
        
        return np.random.uniform(0, 10, N)


    def generate_arms(self,N, T):
        arms = []  

        for i in range(N):
            # Generate an array of size T with normal random variables
            arm = np.random.normal(self.means[i], 1, T)
            arms.append(arm)

        return arms         
    

    
     
    def reset(self):
        self.curr_position = np.zeros(self.N, dtype=int)
        self.rewards = np.zeros(self.T+1)

        self.curr = 0

    def proportions(self): 

        return self.curr_position/self.T 
    

    def next(self, i):
        """

            This function takes i as a parameter and produces the next reward of arm i 

        """
        self.curr_position[i]+=1
        reward = self.arms[i][self.curr_position[i]]
        self.curr+=1 
        self.rewards [self.curr] =  self.rewards [self.curr-1]+reward
        self.averages[i] += (reward-self.averages[i])/self.curr

        
    def curr_arm(self, i):

        return self.curr_position[i]



Class EpsilonGreedyPolicy 

is initiated by supplying the parameter epsilon of $\varepsilon-$greedy policy and an instance bandit of class Bandit.

run(): creates an episode of size $T$ applying the $\varepsilon-$greedy policy and producing  corresponding rewards, which are stores in bandit. Applying the policy it updates the averages of each arm. 

In [ ]:
class EpsilonGreedyPolicy:


    def __init__(self, epsilon, bandit):
        """
            N - number of arms in the bandit problem
            epsilon - parameter of epsilon greedy policy
            bandit - an instance of class Bandit containing rewards 
        """    
        self.N = bandit.N 
        self.epsilon = epsilon  
        self.bandit = bandit 
      

    def run(self):
        """
            applies the policy 
        """
        
        for t in range(1,self.bandit.T):

            u = np.random.uniform(0, 1)
            if (u< self.epsilon):
                arm = np.random.randint(0, self.N)
            else:    
                max_value = np.max(self.bandit.averages)
                arm1 = np.where(self.bandit.averages == max_value)[0]
                arm = arm1[0]
            self.bandit.next(arm)
            


class UCBPolicy is initiated by supplying parameter $c$ of UCB Policy. 

run(): creates an episode of size $T$ applying the UCB policy and producing  corresponding rewards, which are stores in bandit. Applying the policy it updates the averages of each arm. 


In [ ]:
class UCBPolicy:


    def __init__(self, c, bandit):
        """
            N - number of arms in the bandit problem
            epsilon - parameter of UCB policy
            average - array of averages of arm i. 
            bandit - an instance of class Bandit containing rewards 
        """    
        self.N = bandit.N  
        self.c=c 
        self.bandit = bandit 
      

    def run(self):
        """
            applies the policy 
        """
        #play each arm once initially
        for t in range(self.N):
            self.bandit.next(t)
            
        #pick a new arm according to the UCB policy
        for t in range(self.N,self.bandit.T):
            self.epsilon = np.zeros(self.N)
            for a in range(self.N):
                self.epsilon[a]=self.bandit.averages[a]+sqrt(self.c*log(2*t)/self.bandit.curr_position[a])
            max_epsilon = np.max(self.epsilon)
            arm1 = np.where(self.epsilon == max_epsilon)[0]
            arm = arm1[0]
            self.bandit.next(arm)
        


In [ ]:


def main():
    
    T = 1000
    N = 8 
    
    bandit = Bandit(T,N)
    print("The real means of underlying variables are ")
    print(bandit.means)
    
    max_mean = np.max(bandit.means)
    print("The highest mean is ", max_mean , " at arm",np.where(bandit.means == max_mean)[0][0])
    plt.plot(max_mean*np.ones(T), label="Maximal reward")
    
    for epsilon in [0.05, 0.1, 0.2, 0.4]:
        epsilon_greedy_policy = EpsilonGreedyPolicy(epsilon,bandit)
        epsilon_greedy_policy.run()
        print("Proportion of times each arm was pulled in the policy when epsilon =", epsilon)
        print(bandit.proportions())
        average_reward = np.zeros(T)
        for l in range(1,T):
            average_reward[l]=bandit.rewards[l]/l 
        plt.plot(average_reward, label="epsilon-greed with "+str(epsilon))
        print("Sample means of underlying variables are ")
        print(bandit.averages)
   
        bandit.reset()

    for c in [0.001,2,8]:
        UCB_policy = UCBPolicy(c,bandit)
        UCB_policy.run()
        print("Proportion of times each arm was pulled in the policy when c =", c)
        print(bandit.proportions())
        average_reward = np.zeros(T)
        for l in range(1,T):
            average_reward[l]=bandit.rewards[l]/l 
        plt.plot(average_reward, label="UCB with c="+str(c))
        print("Sample means of underlying variables are ")
        print(bandit.averages)
   
        bandit.reset()


    # Plot cumulative rewards
    plt.xlabel('Time')
    plt.ylabel('Reward')
    plt.title('Average rewards')
    plt.legend()
    #plt.text(0,2200,'Maximal possible reward = '+ str(int(T*max_mean)))
    
    plt.show()

# Check if the script is being run directly by Python (not imported as a module)
if __name__ == "__main__":
    # Call the main function
    main()

It is clear from the graphs below that UCB outperforms $\varepsilon$-greedy algorithm. It approaches the maximal reward faster and stays closer to from  early times. Further evidence is given by proportions of times an optimal arm has been pulled - this proportion is higher for UCB. 